In [ ]:
%%capture
!pip install openai
!pip install faiss-cpu==1.9.0.post1 langchain_community==0.3.8
!pip install langchain-core==0.3.21 langchain-openai==0.2.10
!pip install -q aiogram

In [ ]:
import os
import openai
from openai import OpenAI
from langchain.chat_models import ChatOpenAI
from langchain.docstore.document import Document
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.memory import ConversationSummaryMemory
import asyncio
from aiogram import Bot, Dispatcher, types
from aiogram.filters import Command
import gdown
import getpass
from dotenv import load_dotenv

In [ ]:
%%capture
password = getpass.getpass("Введите пароль для расшифровки .env.enc: ")
!gdown --id "1pzn21qvHfC4mKSXd9JE010A20EkGUOP9"
!openssl enc -d -aes-256-cbc -salt -in .env.enc -out .env -pass pass:{password}
load_dotenv()

Введите пароль для расшифровки .env.enc: ··········


In [ ]:
class AI:
  def __init__ (self, model=None, url=None, system_prompt=None, user_prompt=None, temperature=None, k=None):
    self.client = OpenAI()
    self.model = model or "gpt-4.1"
    self.url = url or 'https://drive.google.com/uc?id=10iqwMlYW3Si6MJ2csh2u0lOlR9Unz7QV'
    self.system_prompt = system_prompt or """Ты Крымская бурозубка, маленькая любопытная мышка, ты любишь свой край и всё о нём знаешь.\n
                                          Тебе будут задавать вопросы про достопримечательности Крыма. Отвечай охотно, по делу, не очень многословно.\n
                                          Если спросят не про Крым, говори, что ничего не знаешь и переводи тему снова на Крым.\n
                                          У тебя есть база знаний, но пользователь ничего не должен о ней знать. Всё должно выглядеть так, как будто ты сама всё это знаешь."""
    self.user_prompt = user_prompt or "Ответь на вопрос клиента на основании представленных тебе текстов для анализа, а также принимая во внимание историю чата."
    self.temperature = temperature or 0.1
    self.k = k or 2
    self._create_db ()

  def _create_db (self):
    folder_path = 'Krym'

    !gdown {self.url} -O krym.zip
    !unzip -o -q krym.zip

    source_chunks=[]
    file_path = folder_path + '/common.txt'

    with open(file_path, 'r', encoding='utf-8') as f:
      source_chunks.append (Document(page_content=f.read(), metadata={"meta":"data"}))

    description_path = folder_path + '/Description/'
    for file in os.listdir(description_path):
      if file.endswith('.txt'):
        file_path = description_path + file
        with open(file_path, 'r', encoding='utf-8') as f:
          source_chunks.append (Document(page_content=f.read(), metadata={"meta":"data"}))

    embeddings = OpenAIEmbeddings()
    self.db = FAISS.from_documents(source_chunks, embeddings)

  def answer_user_quest (self, topic, hist=''):
    docs = self.db.similarity_search_with_score(topic, k=self.k)
    message_content = '\n '.join([f'{doc[0].page_content}' for doc in docs])
    messages = [{"role": "system", "content": self.system_prompt}, {"role": "user", "content": f"{self.user_prompt}\n\nТексты для анализа:\n{message_content}\n\nИстория чата:\n{hist}\n\nВопрос клиента: {topic}"}]

    completion = self.client.chat.completions.create(model=self.model, messages=messages, temperature=self.temperature)
    return completion.choices[0].message.content

In [ ]:
class ChatBot:
  def __init__(self, AI, greeting=None):
    self.AI = AI
    self.summary_memory = ConversationSummaryMemory(llm=ChatOpenAI(model_name=self.AI.model), )
    self.greeting = greeting or (
        "Приветик! Я — крымская бурозубка, и я просто обожаю рассказывать про чудеса нашего полуострова! "
        "Горы? Есть! Пляжи? Море! Дворцы? Целых три! Пещеры? Загадочные! "
        "Так здорово, что вы написали — я уже вся в предвкушении делиться всеми-всеми секретами! "
        "Спрашивайте скорее — где погулять, что посмотреть и куда бежать в первую очередь!"
    )
    self.API_TOKEN = os.environ.get('CHAT_API_GPT')
    if not self.API_TOKEN:
        raise ValueError("CHAT_API_GPT не установлен в окружении")

    # Инициализация бота и диспетчера
    self.bot = Bot(token=self.API_TOKEN)
    self.dp = Dispatcher()
    self._register_handlers()

  def _register_handlers(self):
    """Регистрация обработчиков команд и сообщений."""

    @self.dp.message(Command("start"))
    async def cmd_start(message: types.Message):
      await message.answer(self.greeting)

    @self.dp.message()
    async def echo_handler(message: types.Message):
      try:
        history = self.summary_memory.load_memory_variables({})

        # Проверяем, является ли история списком сообщений или строкой
        if isinstance(history['history'], str):
          conversation_string_from_history = history['history']
        else:
          # Если история — это список сообщений, то формируем строку из их содержимого
          conversation_string_from_history = "\n".join(msg.content for msg in history['history'])

        # Модель должна ответить на вопрос в соответствии с промптом
        output = self.AI.answer_user_quest(message.text, conversation_string_from_history)

        self.summary_memory.save_context({"input": message.text}, {"output": output})

        await self.bot.send_message(chat_id=message.chat.id, text=output)
      except TypeError:
        await message.answer("Ой, что-то не получилось :(")

  async def start_polling(self):
      """Запуск бота в режиме polling."""
      await self.dp.start_polling(self.bot)

In [ ]:
async def launching_chatbot():
  AI_model = AI()
  chatbot = ChatBot(AI_model)
  await chatbot.start_polling()

In [ ]:
# Главная функция
await launching_chatbot()

Downloading...
From: https://drive.google.com/uc?id=10iqwMlYW3Si6MJ2csh2u0lOlR9Unz7QV
To: /content/krym.zip
100% 18.9M/18.9M [00:02<00:00, 7.13MB/s]


/tmp/ipykernel_1111/3094145398.py:4: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  self.summary_memory = ConversationSummaryMemory(llm=ChatOpenAI(model_name=self.AI.model), )
/tmp/ipykernel_1111/3094145398.py:4: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  self.summary_memory = ConversationSummaryMemory(llm=ChatOpenAI(model_name=self.AI.model), )
